# BirdCLEF 2026 Starter

Notebook inicial inspirado na estrutura de `/audio/main.ipynb`, mas adaptado para o desafio BirdCLEF 2026.

## 1. Importacoes e configuracao

In [ ]:
from __future__ import annotations

import os
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
from kagglehub.exceptions import UnauthenticatedError
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_SEED = 42
SAMPLE_RATE = 32000
WINDOW_SECONDS = 5.0
HOP_SECONDS = 2.5
MAX_WINDOWS_PER_FILE = 4
MAX_FILES_PER_CLASS = 30
MIN_SAMPLES_PER_CLASS = 2

np.random.seed(RANDOM_SEED)

## 2. Download do dataset

In [ ]:
try:
    DATASET_DIR = Path(kagglehub.competition_download('birdclef-2026')).resolve()
    print('Path to competition files:', DATASET_DIR)
except UnauthenticatedError:
    raise RuntimeError(
        'Kaggle authentication is required before downloading BirdCLEF 2026. '
        'Run `kagglehub login` or configure your Kaggle credentials first.'
    )

## 3. Funcoes auxiliares

In [ ]:
@dataclass
class AudioSample:
    label: str
    path: Path


def resolve_dataset_dir(dataset_dir: str | Path | None = None) -> Path:
    candidates: list[Path] = []

    if dataset_dir is not None:
        candidates.append(Path(dataset_dir).expanduser())

    env_dir = os.getenv('BIRDCLEF_2026_DIR')
    if env_dir:
        candidates.append(Path(env_dir).expanduser())

    if 'DATASET_DIR' in globals():
        candidates.append(Path(DATASET_DIR))

    candidates.extend([
        Path.cwd() / 'birdclef-2026',
        Path.cwd() / 'data' / 'birdclef-2026',
    ])

    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.exists():
            return resolved

    searched = '\n'.join(f'- {path}' for path in candidates) or '- <none>'
    raise FileNotFoundError(
        'BirdCLEF 2026 dataset directory not found. Checked:\n'
        f'{searched}'
    )


def load_audio_windows(
    audio_path: Path,
    sample_rate: int,
    window_seconds: float,
    hop_seconds: float,
    max_windows: int,
) -> list[np.ndarray]:
    signal, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    signal, _ = librosa.effects.trim(signal, top_db=20)

    if signal.size == 0:
        raise ValueError(f'Empty or invalid audio: {audio_path}')

    window_size = max(1, int(window_seconds * sr))
    hop_size = max(1, int(hop_seconds * sr))

    if signal.size <= window_size:
        return [np.pad(signal, (0, window_size - signal.size))]

    windows: list[np.ndarray] = []
    for start in range(0, signal.size - window_size + 1, hop_size):
        windows.append(signal[start:start + window_size])
        if len(windows) >= max_windows:
            break

    return windows or [signal[:window_size]]


def extract_features_from_signal(signal: np.ndarray, sample_rate: int) -> np.ndarray:
    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=20)
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    chroma = librosa.feature.chroma_stft(y=signal, sr=sample_rate)
    spectral_centroid = librosa.feature.spectral_centroid(y=signal, sr=sample_rate)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sample_rate)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=signal, sr=sample_rate)
    zero_crossing = librosa.feature.zero_crossing_rate(signal)
    rms = librosa.feature.rms(y=signal)

    feature_blocks = [
        mfcc,
        mfcc_delta,
        mfcc_delta2,
        chroma,
        spectral_centroid,
        spectral_bandwidth,
        spectral_rolloff,
        zero_crossing,
        rms,
    ]

    features: list[float] = []
    for block in feature_blocks:
        features.extend(np.mean(block, axis=1))
        features.extend(np.std(block, axis=1))

    return np.asarray(features, dtype=np.float32)


def summarize_dataset(dataset_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None]:
    train_csv = pd.read_csv(dataset_dir / 'train.csv')
    taxonomy_csv = pd.read_csv(dataset_dir / 'taxonomy.csv')

    labels_path = dataset_dir / 'train_soundscapes_labels.csv'
    soundscape_labels = pd.read_csv(labels_path) if labels_path.exists() else None

    print(f'Dataset directory: {dataset_dir}')
    print(f'train.csv rows: {len(train_csv):,}')
    print(f'taxonomy.csv rows: {len(taxonomy_csv):,}')
    print(f"Unique classes in train.csv: {train_csv['primary_label'].nunique():,}")
    print(f"Collections: {train_csv['collection'].value_counts().to_dict()}")
    print(f"Class distribution by taxonomy class: {taxonomy_csv['class_name'].value_counts().to_dict()}")

    if soundscape_labels is not None:
        print(f'train_soundscapes_labels.csv rows: {len(soundscape_labels):,}')
        print(f"Unique labeled soundscapes: {soundscape_labels['filename'].nunique():,}")
    else:
        print('train_soundscapes_labels.csv not found.')

    return train_csv, taxonomy_csv, soundscape_labels


def collect_samples(train_audio_dir: Path, train_df: pd.DataFrame) -> list[AudioSample]:
    subset = (
        train_df[['primary_label', 'filename']]
        .dropna()
        .groupby('primary_label', group_keys=False)
        .head(MAX_FILES_PER_CLASS)
    )

    samples: list[AudioSample] = []
    for row in subset.itertuples(index=False):
        audio_path = train_audio_dir / row.filename
        if audio_path.exists():
            samples.append(AudioSample(label=row.primary_label, path=audio_path))

    return samples


def build_feature_matrix(samples: list[AudioSample]) -> tuple[np.ndarray, np.ndarray]:
    feature_rows: list[np.ndarray] = []
    labels: list[str] = []

    for sample in samples:
        try:
            windows = load_audio_windows(
                sample.path,
                SAMPLE_RATE,
                WINDOW_SECONDS,
                HOP_SECONDS,
                MAX_WINDOWS_PER_FILE,
            )
        except Exception as exc:
            print(f'[warn] Failed to load {sample.path.name}: {exc}')
            continue

        for window in windows:
            feature_rows.append(extract_features_from_signal(window, SAMPLE_RATE))
            labels.append(sample.label)

    if not feature_rows:
        raise RuntimeError('No train_audio files could be converted into features.')

    return np.vstack(feature_rows), np.asarray(labels)


def filter_small_classes(features: np.ndarray, labels: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    counts = Counter(labels)
    keep_labels = {label for label, count in counts.items() if count >= MIN_SAMPLES_PER_CLASS}
    keep_mask = np.asarray([label in keep_labels for label in labels], dtype=bool)
    return features[keep_mask], labels[keep_mask]

## 4. Auditoria inicial do dataset

In [ ]:
dataset_dir = resolve_dataset_dir()
train_df, taxonomy_df, soundscape_labels_df = summarize_dataset(dataset_dir)

train_audio_dir = dataset_dir / 'train_audio'
print('train_audio exists:', train_audio_dir.exists())
print('train_soundscapes exists:', (dataset_dir / 'train_soundscapes').exists())
print('test_soundscapes exists:', (dataset_dir / 'test_soundscapes').exists())

## 5. Baseline simples com train_audio

In [ ]:
samples = collect_samples(train_audio_dir, train_df)
print(f'Audio files selected: {len(samples):,}')
print(f'Classes selected: {len({sample.label for sample in samples}):,}')

features, labels = build_feature_matrix(samples)
features, labels = filter_small_classes(features, labels)

print('Feature matrix shape:', features.shape)
print(f'Window labels kept: {len(labels):,}')

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

x_train, x_val, y_train, y_val = train_test_split(
    features,
    encoded_labels,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=encoded_labels,
)

model = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            multi_class='multinomial',
            random_state=RANDOM_SEED,
        )),
    ]
)

model.fit(x_train, y_train)
predictions = model.predict(x_val)

target_names = label_encoder.inverse_transform(np.unique(y_val))
print(classification_report(
    y_val,
    predictions,
    labels=np.unique(y_val),
    target_names=target_names,
    zero_division=0,
))

## 6. Proximo passo

Este baseline e apenas um ponto de partida. O proximo passo mais importante e usar `train_soundscapes_labels.csv` para treinar previsoes multilabel por janelas de 5 segundos, aproximando melhor a tarefa real da competicao.